In [ ]:
import pandas as pd
import os

INTER_DIR = "../data/02_intermediate/"
DOCS_DIR = "../docs/"
PROCESSED_DIR = "../data/03_processed/"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Starte Data Fusion...")

# ==========================================
# Daten laden
# ==========================================
df_pks = pd.read_csv(os.path.join(INTER_DIR, 'pks_mapped.csv'), sep=';')
df_pks = df_pks[pd.to_numeric(df_pks['jahr'], errors='coerce') > 1900]  # Artefakt-Zeilen entfernen (z.B. Header-Rest mit jahr=3)
df_jus = pd.read_csv(os.path.join(INTER_DIR, 'justiz_mapped.csv'), sep=';')
df_bev = pd.read_csv(os.path.join(INTER_DIR, 'bevoelkerung_mapped.csv'), sep=';')
df_mapping = pd.read_csv(os.path.join(DOCS_DIR, 'mapping_table_autofilled_v2_haendisch.csv'), sep=';')

# ==========================================
# Datenbereinigung: "Gesamt"-Nationalität entfernen
# ==========================================
# "Gesamt" = Deutsch + Nichtdeutsch -> würde bei Aggregation zu Doppelzählung führen
df_pks = df_pks[df_pks['nationalitaet'] != 'Gesamt']
df_jus = df_jus[df_jus['nationalitaet'] != 'Gesamt']
df_bev = df_bev[df_bev['nationalitaet'] != 'Gesamt']

print(f"  PKS nach Filter: {len(df_pks)} Zeilen")
print(f"  Justiz nach Filter: {len(df_jus)} Zeilen")

# ==========================================
# Datenbereinigung: Hierarchische Aggregats-Kategorien entfernen
# ==========================================
# Die Justiz-Daten enthalten Eltern-Zeilen, die Summen ihrer Kinder darstellen.
# Diese müssen raus, um Doppelzählung bei Analysen zu vermeiden.
#
# Entfernt (reine Summenzeilen ohne eigene PKS-Zuordnung):
#   "Straftaten insgesamt", "Straftaten ohne Straftaten im Straßenverkehr",
#   "Straftaten im Straßenverkehr"
#
# Entfernt (Justiz-Unterkategorien, deren Werte im Elternteil D&U enthalten sind):
#   "Diebstahl", "Schwerer Diebstahl"
#   -> "Diebstahl und Unterschlagung" ist das einzige PKS-Mapping-Target für Diebstahl
#
# Beibehalten trotz Aggregats-Charakter (haben eigene PKS-Mappings):
#   "Straftaten nach dem StGB (ohne Verkehr)" (26 PKS-Zuordnungen)
#   "Straftaten gegen die sexuelle Selbstbestimmung" (67 PKS-Zuordnungen)
#   "Straftaten nach anderen Bundes- u. Landesgesetzen" (57 PKS-Zuordnungen)

aggregat_rein = [
    'Straftaten insgesamt',
    'Straftaten ohne Straftaten im Straßenverkehr',
    'Straftaten im Straßenverkehr',
]

justiz_kinder_von_du = [
    'Diebstahl',
    'Schwerer Diebstahl',
]

df_jus = df_jus[~df_jus['straftat_bezeichnung'].isin(aggregat_rein + justiz_kinder_von_du)]
print(f"  Justiz nach Aggregats-Filter: {len(df_jus)} Zeilen")

# ==========================================
# PKS-Delikte auf Justiz-Niveau aggregieren (Many-to-One)
# ==========================================
# Die ~905 PKS-Delikte werden über die Mapping-Tabelle auf 21 Justiz-Kategorien abgebildet.
# Anschließend werden die Tatverdächtigen pro (Jahr, Nationalität, Justiz-Kategorie) aufsummiert.
print("Aggregiere PKS über Mapping-Tabelle...")

df_pks_joined = pd.merge(
    df_pks, df_mapping,
    left_on='straftat_bezeichnung', right_on='pks_straftat_bezeichnung',
    how='inner'
)

df_pks_agg = df_pks_joined.groupby(
    ['jahr', 'nationalitaet', 'justiz_straftat_bezeichnung_mapped']
)['tatverdaechtige'].sum().reset_index()

df_pks_agg.rename(columns={'justiz_straftat_bezeichnung_mapped': 'straftat_bezeichnung'}, inplace=True)

# Reine Aggregats-Kategorien auch aus PKS-Ergebnis entfernen
df_pks_agg = df_pks_agg[~df_pks_agg['straftat_bezeichnung'].isin(aggregat_rein)]
print(f"  PKS aggregiert: {len(df_pks_agg)} Zeilen")

# ==========================================
# Outer Join (PKS + Justiz + Bevölkerung)
# ==========================================
print("Führe Datenquellen zusammen...")

# Outer Join: Behält alle Zeilen, auch wenn eine Quelle für bestimmte Jahre fehlt
df_master = pd.merge(
    df_pks_agg, df_jus,
    on=['jahr', 'nationalitaet', 'straftat_bezeichnung'],
    how='outer'
)

# Bevölkerung per Left Join anhängen (über jahr + nationalitaet)
df_master = pd.merge(
    df_master, df_bev,
    on=['jahr', 'nationalitaet'],
    how='left'
)

# Fehlende Kriminalitätswerte mit 0 auffüllen (z.B. PKS erst ab 1987 vorhanden)
df_master['tatverdaechtige'] = df_master['tatverdaechtige'].fillna(0).astype(int)
df_master['verurteilte'] = df_master['verurteilte'].fillna(0).astype(int)

# ==========================================
# Feature Engineering (Kriminalitätsraten pro 100.000)
# ==========================================
# Rate wird gegen die jeweilige Bevölkerungskohorte gerechnet
# (z.B. Nichtdeutsch-TV gegen Ausländer-Bevölkerung, nicht Gesamtbevölkerung)
print("Berechne Kriminalitätsraten pro 100.000 Einwohner...")
df_master['tatverdaechtige_pro_100k'] = (df_master['tatverdaechtige'] / df_master['bevoelkerung_gesamt']) * 100000
df_master['verurteilte_pro_100k'] = (df_master['verurteilte'] / df_master['bevoelkerung_gesamt']) * 100000

# Finalen Datensatz speichern
output_path = os.path.join(PROCESSED_DIR, 'master_kriminalitaet_1984_2024.csv')
df_master.to_csv(output_path, index=False, sep=';')

print(f"\nData Fusion abgeschlossen.")
print(f"Gespeichert: {output_path}")
print(f"Zeilen: {len(df_master)}, Kategorien: {df_master['straftat_bezeichnung'].nunique()}, Nationalitäten: {df_master['nationalitaet'].unique().tolist()}")
display(df_master.sample(5))